# Avoiding full dense reconstruction
Profiling the kl_factor_update_largedims function has shown that memory keeps blowing up at
We will try to avoid that here by only reconstructing an explicit sparse copy

In [1]:
import tensorly as tl
import pytensorlab as ptl

from tensorly.tucker_tensor import validate_tucker_rank
from tensorly.tenalg import mode_dot

from tensormet.sparse_ops import (
    unfold_from_vectorized_sparse,
    ptl_tucker_to_tensor,
)
from tensormet.utils import ThreadBudget, DATA_DIR, compute_num_threads, select_gpu
from tensormet.tucker_tensor import SparseTupleTensor
from tensormet.config import ExperimentConfig, TrainingConfig, EvalConfig, RunConfig
from tensormet.sparse_ops import initialize_nonnegative_tucker

from memory_profiler import profile
import os
import pickle
from pathlib import Path
import time
import numpy as np

device = select_gpu()

tl.set_backend("cupy")

2026-02-11 12:27:21.620614: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


Selected GPU 3 with 454.62 MB used memory.


In [2]:
dataset = "fineweb-en"
dim = 20000
method = "siiSoftPlus"
divergence = "fr"
name = "profile"
random_state = 1
rank = [150, 150, 150]
max_cpu_frac = 0.66
iters = 500
tol = 1e-5
patience = 5
rec_check_every = 1
rec_log_every = 1
sem_check_every = 2
sem_error_type = "absolute_prob_score"
max_cpu_frac = 0.66
thread_budget = ThreadBudget(n_threads=compute_num_threads(max_cpu_frac))

In [3]:
vocab_path = os.path.join(DATA_DIR, "tensors", dataset, f"vocabularies/{dim}.pkl")
with open(vocab_path, "rb") as f:
    vocab = pickle.load(f)
cfg = RunConfig(
    exp=ExperimentConfig(
        dataset=dataset,
        method=method,
        divergence=divergence,
        dim=dim,
        rank=tuple(rank),
        name=name,
        random_state=random_state,
        max_cpu_frac=max_cpu_frac,
        data_dir=Path(DATA_DIR),
    ),
    train=TrainingConfig(
        n_iter_max=iters,
        tol=tol,
        epsilon=1e-12,
        warmup_steps=1 if divergence == "kl" else 50,
        patience=patience,
        normalize_factors=False,
        verbose=True,
        return_errors="full",
    ),
    eval=EvalConfig(
        rec_log_every=rec_log_every,
        rec_check_every=rec_check_every,
        sem_check_every=sem_check_every,
        sem_error_type=sem_error_type,
        remove_OOV=False,
    ),
)
print(cfg)
paths = cfg.artifact_paths()
for p in paths.values():
    if isinstance(p, Path):
        p.parent.mkdir(parents=True, exist_ok=True)



RunConfig(exp=ExperimentConfig(dataset='fineweb-en', method='siiSoftPlus', divergence='fr', dim=20000, rank=(150, 150, 150), name='profile', random_state=1, max_cpu_frac=0.66, data_dir=PosixPath('/home/local/stefa/data')), train=TrainingConfig(n_iter_max=500, tol=1e-05, epsilon=1e-12, init='random', normalize_factors=False, warmup_steps=50, patience=5, verbose=True, return_errors='full'), eval=EvalConfig(rec_check_every=1, rec_log_every=1, sem_check_every=2, sem_error_type='absolute_prob_score', sem_softmax_temperature=0.1, sem_fitness_target=10000, n_sentence_cache=None, remove_OOV=False, time_iteration=True))


In [4]:
sparse_tensor = SparseTupleTensor.load_from_disk(
    dataset=dataset,
    method=method,
    dims=dim,
    map_location="cpu",
)
sparse_tensor.tensor_to_sparse("cupy")

In [5]:
shape = tuple(sparse_tensor.shape)
rank = validate_tucker_rank(shape, rank=rank)
modes = list(range(len(rank)))
init = cfg.train.init
core, factors = initialize_nonnegative_tucker(sparse_tensor.tensor, shape, rank, modes, init, random_state)

In [6]:
# -- Frobenius Norm --
from tensorly.tucker_tensor import tucker_to_tensor
from tensorly.base import unfold
def fr_factor_update(vec_tensor, core, factors, mode, shape, thread_budget=None, epsilon=1e-12):
    # This still explodes! The created Z tensor does not always fit in memory
    X = unfold_from_vectorized_sparse(vec_tensor, shape,
                                             mode)  # this is the same as B when using dense tensor!
    Z = tucker_to_tensor((core, factors), skip_factor=mode)
    Z = tl.transpose(unfold(Z, mode))
    numerator = X @ Z  # cupy sparse @ dense
    numerator = tl.clip(numerator, a_min=epsilon, a_max=None)
    A = factors[mode]
    denominator = tl.dot(A, tl.dot(tl.transpose(Z), Z))
    denominator = tl.clip(denominator, a_min=epsilon, a_max=None)
    A *= numerator / denominator
    A = tl.clip(A, a_min=epsilon, a_max=None)
    return A

In [7]:
import cupy as cp

# force CUDA context init
cp.cuda.Device(0).use()
_ = cp.zeros(1)

# basic info
print("CuPy:", cp.__version__)
print("Device:", cp.cuda.runtime.getDeviceProperties(0)["name"].decode())
print("CUDA runtime:", cp.cuda.runtime.runtimeGetVersion())
print("Driver:", cp.cuda.runtime.driverGetVersion())

CuPy: 13.6.0
Device: NVIDIA GeForce RTX 3090
CUDA runtime: 12090
Driver: 12090


In [8]:
import cupyx.scipy.sparse as cpx_sparse
import sparse
mode = 0
start_time = time.time()
factor = fr_factor_update(
                    vec_tensor=sparse_tensor.tensor,
                    core=core,
                    factors=factors,
                    mode=mode,
                    shape=shape,
                    thread_budget=thread_budget)
print(f"calculated in {time.time()-start_time} seconds")

OutOfMemoryError: Out of memory allocating 240,000,000,000 bytes (allocated so far: 3,713,036,288 bytes).

In [9]:
import cupy as cp
from tensormet.sparse_ops import unfold_from_vectorized_sparse
from tensormet.utils import einsum_letters
from tensormet.distance import _unravel_cols_for_mode as tunravelcolformod
from tensormet.distance import _compute_Zcols_batch as tcompZcolbatch
def _unravel_cols_for_mode(cols, shape, mode):
    """
    Convert unfolding column indices -> per-mode indices for all modes != `mode`,
    assuming C-order flattening where the last remaining mode varies fastest.
    """
    N = len(shape)
    other_modes = [m for m in range(N) if m != mode]
    other_dims = [shape[m] for m in other_modes]

    u = cols
    idxs_rev = []
    for dim in reversed(other_dims):
        idxs_rev.append(u % dim)
        u = u // dim

    idxs = list(reversed(idxs_rev))
    return other_modes, {m: idxs[i] for i, m in enumerate(other_modes)}


def _compute_Zcols_batch(core, factors, mode, other_modes, idxs_by_mode, epsilon=1e-12):
    """
    Compute Z columns (as rows) for a batch of unfolding columns, without building full Z.

    Returns Z_u with shape (m, R_mode), where m = batch size.
    """
    N = core.ndim
    letters = einsum_letters(N)
    core_subs = "".join(letters)

    # factor-row matrices for each other mode: (m, Rk)
    mats = [factors[k][idxs_by_mode[k]] for k in other_modes]

    # einsum: core[a b c ...], M_b[m b], M_c[m c], ... -> out[m a_mode]
    in_terms = [core_subs] + [("m" + letters[k]) for k in other_modes]
    out_term = "m" + letters[mode]
    eq = ",".join(in_terms) + "->" + out_term

    Z_u = cp.einsum(eq, core, *mats)
    Z_u = cp.clip(Z_u, a_min=epsilon, a_max=None)
    return Z_u


def _core_unfold(core, mode):
    """
    Mode-n unfolding of the Tucker core: (R_mode, prod(other Rk))
    using C-order flattening with remaining modes in increasing order.
    """
    G = cp.moveaxis(core, mode, 0)
    return G.reshape(G.shape[0], -1)


def _tucker_gram_ZtZ(core, factors, mode, epsilon=1e-12):
    """
    Compute Gram = Z^T Z exactly, without forming Z.

    In your dense version:
        Z = transpose(unfold(tucker_to_tensor(skip_factor=mode), mode))  # (J, R_mode)
        Gram = Z^T Z                                                    # (R_mode, R_mode)

    Algebra:
        Z = K @ G_(mode)^T
        with K = kron_{k!=mode}(A_k)  and  G_(mode) is core unfolded along mode.

        K^T K = kron_{k!=mode}(A_k^T A_k)

        Gram = G_(mode) @ (K^T K) @ G_(mode)^T

    We compute this by contracting the core with the per-mode Gram matrices (A_k^T A_k),
    then doing one small matrix multiply in rank-space.
    """
    N = core.ndim
    letters = einsum_letters(2 * N)  # need “primed” letters too
    base = letters[:N]
    prim = letters[N:2 * N]

    core_subs = "".join(base)

    other_modes = [k for k in range(N) if k != mode]
    grams = [factors[k].T @ factors[k] for k in other_modes]

    # Build output subscripts: keep mode letter, replace others by their primed version
    out = list(base)
    for k in other_modes:
        out[k] = prim[k]
    out_subs = "".join(out)

    # core[a b c ...], G_b[b B], G_c[c C], ... -> out[a B C ...] (mode stays unprimed)
    gram_terms = [f"{base[k]}{prim[k]}" for k in other_modes]
    eq = core_subs + "," + ",".join(gram_terms) + "->" + out_subs

    Gp = cp.einsum(eq, core, *grams)  # same shape as core, but other modes live in primed space

    G_unf = _core_unfold(core, mode)   # (R_mode, P)
    Gp_unf = _core_unfold(Gp, mode)    # (R_mode, P)

    Gram = Gp_unf @ G_unf.T            # (R_mode, R_mode) -> sparse
    Gram = cp.clip(Gram, a_min=epsilon, a_max=None)
    return Gram


def fr_factor_update_nodense_equiv(
    vec_tensor,
    core,
    factors,
    mode,
    shape,
    epsilon=1e-12,
    batch_cols=20000,
):
    """
    Frobenius (Euclidean) multiplicative update for Tucker factor A^(mode)
    WITHOUT building dense Z, but equivalent to your dense function 3:

        numerator   = X @ Z
        denominator = A @ (Z^T Z)
        A <- A * numerator / denominator

    where X is the sparse unfolding and Z = transpose(unfold(tucker_to_tensor(skip_factor=mode), mode)).
    """
    # Sparse unfolding X_(mode)
    X = unfold_from_vectorized_sparse(vec_tensor, shape, mode).tocoo()
    rows = X.row
    cols = X.col
    vals = X.data

    A = factors[mode]  # (I_mode, R_mode)

    # ---- Denominator part: Gram = Z^T Z exactly, no Z materialization
    Gram = _tucker_gram_ZtZ(core, factors, mode, epsilon=epsilon)  # (R, R)
    denominator = A @ Gram
    denominator = cp.clip(denominator, a_min=epsilon, a_max=None)

    # ---- Numerator part: numerator = X @ Z via batching unique columns, no full Z
    numerator = cp.zeros_like(A)

    ucols, inv = cp.unique(cols, return_inverse=True)
    other_modes = [m for m in range(len(shape)) if m != mode]

    for start in range(0, int(ucols.size), int(batch_cols)):
        end = min(start + int(batch_cols), int(ucols.size))
        u = ucols[start:end]

        _, idxs_by_mode = _unravel_cols_for_mode(u, shape, mode)

        # Z_u: (m, R_mode)  where row t is Z[column=u[t], :]
        Z_u = _compute_Zcols_batch(
            core=core,
            factors=factors,
            mode=mode,
            other_modes=other_modes,
            idxs_by_mode=idxs_by_mode,
            epsilon=epsilon,
        )

        # nnz entries belonging to these unique columns
        nz_idx = cp.where((inv >= start) & (inv < end))[0]
        if nz_idx.size == 0:
            continue

        r_i = rows[nz_idx]          # (nnz_b,)
        v_i = vals[nz_idx]          # (nnz_b,)
        u_i = inv[nz_idx] - start   # local index into this batch [0..m)

        Z_rows = Z_u[u_i]           # (nnz_b, R_mode)

        # numerator[row] += X_ij * Z[j,:]
        cp.add.at(numerator, r_i, v_i[:, None] * Z_rows)

    # MU update
    A_new = A * (numerator / (denominator + 1e-12))
    A_new = cp.clip(A_new, a_min=epsilon, a_max=None)
    return A_new
def fr_factor_update_nodense_equiv_bis(
    vec_tensor,
    core,
    factors,
    mode,
    shape,
    epsilon=1e-12,
    batch_cols=20000,
):
    """
    Frobenius (Euclidean) multiplicative update for Tucker factor A^(mode)
    WITHOUT building dense Z, but equivalent to your dense function 3:

        numerator   = X @ Z
        denominator = A @ (Z^T Z)
        A <- A * numerator / denominator

    where X is the sparse unfolding and Z = transpose(unfold(tucker_to_tensor(skip_factor=mode), mode)).
    """
    # Sparse unfolding X_(mode)
    X = unfold_from_vectorized_sparse(vec_tensor, shape, mode).tocoo()
    rows = X.row
    cols = X.col
    vals = X.data

    A = factors[mode]  # (I_mode, R_mode)

    # ---- Denominator part: Gram = Z^T Z exactly, no Z materialization
    Gram = _tucker_gram_ZtZ(core, factors, mode, epsilon=epsilon)  # (R, R)
    denominator = A @ Gram
    denominator = cp.clip(denominator, a_min=epsilon, a_max=None)

    # ---- Numerator part: numerator = X @ Z via batching unique columns, no full Z
    numerator = cp.zeros_like(A)

    ucols, inv = cp.unique(cols, return_inverse=True)
    other_modes = [m for m in range(len(shape)) if m != mode]

    for start in range(0, int(ucols.size), int(batch_cols)):
        end = min(start + int(batch_cols), int(ucols.size))
        u = ucols[start:end]

        _, idxs_by_mode = tunravelcolformod(u, shape, mode)

        # Z_u: (m, R_mode)  where row t is Z[column=u[t], :]
        Z_u = tcompZcolbatch(
            core=core,
            factors=factors,
            mode=mode,
            other_modes=other_modes,
            idxs_by_mode=idxs_by_mode,
            epsilon=epsilon,
        )

        # nnz entries belonging to these unique columns
        nz_idx = cp.where((inv >= start) & (inv < end))[0]
        if nz_idx.size == 0:
            continue

        r_i = rows[nz_idx]          # (nnz_b,)
        v_i = vals[nz_idx]          # (nnz_b,)
        u_i = inv[nz_idx] - start   # local index into this batch [0..m)

        Z_rows = Z_u[u_i]           # (nnz_b, R_mode)

        # numerator[row] += X_ij * Z[j,:]
        cp.add.at(numerator, r_i, v_i[:, None] * Z_rows)

    # MU update
    A_new = A * (numerator / (denominator + 1e-12))
    A_new = cp.clip(A_new, a_min=epsilon, a_max=None)
    return A_new



In [10]:
mode = 0
start_time = time.time()
factor_nodense = fr_factor_update_nodense_equiv(
                    vec_tensor=sparse_tensor.tensor,
                    core=core,
                    factors=factors,
                    mode=mode,
                    shape=shape,)
print(f"calculated in {time.time()-start_time} seconds")
start_time = time.time()
factor_nodense_bis = fr_factor_update_nodense_equiv_bis(
                    vec_tensor=sparse_tensor.tensor,
                    core=core,
                    factors=factors,
                    mode=mode,
                    shape=shape,)
print(f"calculated in {time.time()-start_time} seconds")


calculated in 1.874910831451416 seconds
calculated in 1.7856714725494385 seconds


In [11]:
for el in [factor, factor_nodense, factor_nodense_bis]:
    for comp in [factor, factor_nodense, factor_nodense_bis]:
        print(cp.allclose(el, comp))

NameError: name 'factor' is not defined

# Core calculations

In [12]:
from tensormet.sparse_ops import(
    gather_dense_at_block_nz,
    sparse_multi_mode_dot_vec
)

In [13]:
def fr_core_update(vec_tensor, shape, core, factors, modes, thread_budget=None, epsilon=1e-12):
    """
    One multiplicative KL update for the core tensor.

    [DESCRIBE]

    Returns
    -------
    core : updated core.
    """

    numerator = sparse_multi_mode_dot_vec(
        vec_tensor=vec_tensor,
        orig_shape=shape,
        factors=factors,
        modes=modes,
        transpose_factors=True,  # X ×_n W_n^T
    )
    # we clip the numerator
    numerator = tl.clip(numerator, a_min=epsilon, a_max=None)
    # these operations can again be done with the dense implementation
    for i, f in enumerate(factors):
        if i:
            denominator = mode_dot(denominator, tl.dot(tl.transpose(f), f), i)
        else:
            denominator = mode_dot(core, tl.dot(tl.transpose(f), f), i)
    denominator = tl.clip(denominator, a_min=epsilon, a_max=None)

    new_core = core * numerator / (denominator + epsilon)
    return new_core

In [14]:
core_old = fr_core_update(vec_tensor=sparse_tensor.tensor,
                    core=core,
                    factors=factors,
                    modes=modes,
                    shape=shape,
                    thread_budget=thread_budget,)

OutOfMemoryError: Out of memory allocating 240,000,000,000 bytes (allocated so far: 1,910,391,296 bytes).

In [15]:
from tqdm import tqdm
import cupy as cp
import numpy as np

def _gpu_free_bytes():
    """
    Conservative 'free bytes now' estimate.
    - memGetInfo: free bytes in the CUDA context (driver view)
    - default mempool: may have cached blocks; it can still grow if driver has free
    We return driver-free (most conservative for new allocations).
    """
    free_b, total_b = cp.cuda.runtime.memGetInfo()
    return int(free_b)


def estimate_batch_num_for_outer(
    factors,
    core,
    safety=0.50,
    temp_mult=1.5,
):
    """
    Much more conservative batch estimate for _accumulate_core_num_outer,
    because it often materializes intermediate outer-products / broadcasts.

    This is intentionally pessimistic: uses prod(R) scaling as a worst-case.
    """
    N = len(factors)
    dtype = core.dtype

    itemsize = int(np.dtype(dtype).itemsize)
    R = [int(factors[n].shape[1]) for n in range(N)]
    core_size = int(np.prod(R))

    # worst-case: per-b element touches/temporaries proportional to core_size
    # (many implementations end up with something like (b, core_size) transiently)
    bytes_per_b = core_size * itemsize

    # plus gathered mats and indices (usually small compared to core_size, but add anyway)
    bytes_per_b += sum(R) * itemsize + N * 8
    bytes_per_b = int(np.ceil(bytes_per_b * temp_mult))

    free_b = _gpu_free_bytes()
    budget_b = int(free_b * safety)

    b = max(1, budget_b // max(1, bytes_per_b))
    return int(b)


def estimate_batch_rhat_for_tensordot(
    core,
    factors,
    safety=0.60,
    temp_mult=1.3,     # tensordot may use extra workspace; keep >1
):
    """
    Estimate safe batch size for _rhat_from_factor_rows_sequential that starts with:
        tmp = tensordot(mats[0], core, axes=(1,0))  -> (b, prod(R[1:]))
    """
    N = len(factors)
    R = [int(factors[n].shape[1]) for n in range(N)]

    dtype = core.dtype
    itemsize = int(np.dtype(dtype).itemsize)

    # Dominant allocation: tmp has (b, prod(R[1:])) elements
    prod_rest = int(np.prod(R[1:])) if N > 1 else 1
    tmp_bytes_per_b = prod_rest * itemsize

    # Also gather mats: sum_n (b*Rn) elements
    mats_bytes_per_b = sum(R) * itemsize

    # Small vectors (r_hat, w) and idx slices (negligible, but include)
    small_bytes_per_b = (4 * itemsize) + (N * 8)

    bytes_per_b = int(np.ceil((tmp_bytes_per_b + mats_bytes_per_b + small_bytes_per_b) * temp_mult))

    free_b = _gpu_free_bytes()
    budget_b = int(free_b * safety)

    b = budget_b // max(1, bytes_per_b)
    return int(b)

# --- reuse from your function B (unchanged) ---
def _accumulate_core_num_outer(Num, w, mats):
    """
    Num: core-shaped accumulator (R0,...,R_{N-1})
    w  : (b,)
    mats[n]: (b, Rn)

    Updates Num in-place: Num += sum_m w[m] * outer(mats0[m], mats1[m], ...)
    """
    b = w.shape[0]
    N = len(mats)

    T = w[:, None] * mats[0]  # (b, R0)

    for n in range(1, N):
        T = T[..., None] * mats[n].reshape((b,) + (1,) * (T.ndim - 1) + (mats[n].shape[1],))

    Num += cp.sum(T, axis=0)


def _blocked_coo_to_flat_indices(vec_tensor, orig_shape):
    orig_shape = tuple(orig_shape)
    size = int(np.prod(orig_shape))
    int32_max = np.iinfo(np.int32).max
    block_size = min(size, int32_max)

    coo = vec_tensor.tocoo()
    flat = coo.row.astype(cp.int64) + coo.col.astype(cp.int64) * cp.int64(block_size)
    vals = coo.data
    return flat, vals


def _unravel_flat_indices_C(flat, shape):
    shape = tuple(int(s) for s in shape)
    u = flat
    idxs_rev = []
    for dim in reversed(shape):
        idxs_rev.append(u % dim)
        u = u // dim
    return list(reversed(idxs_rev))


# --- new small helper: denominator in rank space, core-sized only ---
def _core_multilinear_grams(core, grams, epsilon=1e-12):
    """
    Compute:
        D = core ×_0 grams[0] ×_1 grams[1] × ... ×_{N-1} grams[N-1]
    where grams[n] = A_n^T A_n has shape (R_n, R_n).

    Returns D with the same shape as core, without mode_dot / tl overhead.
    """
    tmp = core
    N = core.ndim
    for n in range(N):
        G = grams[n]  # (R_n, R_n)
        # tensordot over core axis n: (R_n,R_n) x (...,R_n,...) -> (R_n, ..., ...)
        tmp = cp.tensordot(G, tmp, axes=(1, n))
        # tensordot brings the new R_n axis to the front; move it back to position n
        tmp = cp.moveaxis(tmp, 0, n)
    tmp = cp.clip(tmp, a_min=epsilon, a_max=None)
    return tmp


def fr_core_update_norecon(
    vec_tensor,
    shape,
    core,
    factors,
    modes=None,              # assumes all modes (same constraint style as your KL v2)
    thread_budget=None,      # kept for API compatibility
    epsilon=1e-12,
    batch_num=64,
):
    """
    Frobenius (Euclidean) multiplicative update for the Tucker core WITHOUT dense recon.

    Equivalent to your original function C:
        numerator   = X ×_n A_n^T
        denominator = core ×_n (A_n^T A_n)
        core *= numerator / denominator

    but numerator is accumulated by streaming NNZ and building only core-sized tensors.
    """
    shape = tuple(int(s) for s in shape)
    N = len(shape)

    if modes is None:
        modes = list(range(N))
    if list(modes) != list(range(N)):
        raise NotImplementedError("This version assumes modes == all modes (0..N-1).")

    # --- decode NNZ coordinates (same approach as KL core v2) ---
    flat, xvals = _blocked_coo_to_flat_indices(vec_tensor, shape)
    nnz = int(flat.size)
    if nnz == 0:
        return core

    idxs = _unravel_flat_indices_C(flat, shape)  # list length N, each (nnz,)

    # --- numerator: core-shaped accumulator, streamed over NNZ ---
    Num = cp.zeros_like(core)
    print("starting batch")
    # small batches keep peak memory down (like your pass-2 accumulator)
    for start in tqdm(range(0, nnz, int(batch_num))):
        end = min(start + int(batch_num), nnz)
        mats = [factors[n][idxs[n][start:end]] for n in range(N)]  # each (b, Rn)
        w = xvals[start:end]  # Frobenius numerator uses X directly (no X/R like KL)
        _accumulate_core_num_outer(Num, w, mats)

    Num = cp.clip(Num, a_min=epsilon, a_max=None)

    # --- denominator: rank-space multilinear product with Gram matrices ---
    grams = [factors[n].T @ factors[n] for n in range(N)]  # each (R_n, R_n)
    Den = _core_multilinear_grams(core, grams, epsilon=epsilon)  # core-shaped

    # --- MU update ---
    core_new = core * (Num / (Den + epsilon))
    return core_new


In [16]:
batch_num = estimate_batch_num_for_outer(factors, core, safety=0.99)
print(batch_num)
core_new = fr_core_update_norecon(vec_tensor=sparse_tensor.tensor,
                    core=core,
                    factors=factors,
                    modes=modes,
                    shape=shape,
                    thread_budget=thread_budget,
                    batch_num=batch_num,
                    )

1128
starting batch


100%|██████████| 2699/2699 [02:27<00:00, 18.34it/s] 


In [17]:
core

array([[[0.427022  , 0.7303245 , 0.01011437, ..., 0.72298896,
         0.569717  , 0.02255598],
        [0.08197428, 0.9772763 , 0.57810044, ..., 0.08962608,
         0.9928171 , 0.19161285],
        [0.8218587 , 0.88496166, 0.69841325, ..., 0.9252743 ,
         0.15555823, 0.16773006],
        ...,
        [0.7639514 , 0.56936574, 0.13512744, ..., 0.0473403 ,
         0.44560006, 0.72389215],
        [0.6271814 , 0.40939045, 0.05514596, ..., 0.23230018,
         0.83685964, 0.8842653 ],
        [0.3765482 , 0.35131177, 0.6091856 , ..., 0.38850707,
         0.3878181 , 0.4017717 ]],

       [[0.7110517 , 0.35874972, 0.90712786, ..., 0.9932229 ,
         0.998264  , 0.19537595],
        [0.9103249 , 0.07662821, 0.2673538 , ..., 0.8389651 ,
         0.7035953 , 0.13838407],
        [0.3528635 , 0.48088828, 0.8490924 , ..., 0.2973331 ,
         0.9504827 , 0.32120535],
        ...,
        [0.86513823, 0.8284465 , 0.04706743, ..., 0.01778889,
         0.7805686 , 0.19786221],
        [0.0

In [18]:
core_old

NameError: name 'core_old' is not defined

In [19]:
core_new

array([[[3.20939374e-13, 5.80475988e-13, 9.02531732e-15, ...,
         5.83954380e-13, 4.69337188e-13, 1.84522638e-14],
        [5.89604591e-14, 7.45792371e-13, 4.95718917e-13, ...,
         6.94593038e-14, 7.85043254e-13, 1.50361684e-13],
        [6.41822370e-13, 7.30850981e-13, 6.43075057e-13, ...,
         7.74095143e-13, 1.32744944e-13, 1.42143296e-13],
        ...,
        [6.51657927e-13, 5.11050132e-13, 1.34314109e-13, ...,
         4.30549930e-14, 4.12938347e-13, 6.65887755e-13],
        [4.51475690e-13, 3.12569062e-13, 4.72747079e-14, ...,
         1.79948843e-13, 6.61604343e-13, 6.94149057e-13],
        [2.96272528e-13, 2.91831555e-13, 5.64373906e-13, ...,
         3.27148627e-13, 3.33042784e-13, 3.42753874e-13]],

       [[5.06045509e-13, 2.70743090e-13, 7.65041947e-13, ...,
         7.63148442e-13, 7.77800946e-13, 1.51507116e-13],
        [6.18139925e-13, 5.53960022e-14, 2.16103014e-13, ...,
         6.16965300e-13, 5.24695467e-13, 1.02731388e-13],
        [2.59740092e-13, 

In [21]:
core_new.min()

array(4.0244907e-15, dtype=float32)

In [195]:
cp.allclose(core_old, core_new)

array(True)

# Error calculation


In [224]:
import cupy as cp
import numpy as np
from tensormet.distance import _rhat_from_factor_rows_sequential as trhat
# Assumes you already have these from your previous shifts:
# - _blocked_coo_to_flat_indices(vec_tensor, shape)
# - _unravel_flat_indices_C(flat, shape)
# - _rhat_from_factor_rows_sequential(core, mats, epsilon=1e-12)

def _core_multilinear_grams(core, grams, epsilon=1e-12):
    """
    D = core ×_0 grams[0] ×_1 grams[1] × ... ×_{N-1} grams[N-1]
    where grams[n] = A_n^T A_n (R_n, R_n). Returns core-shaped D.
    """
    tmp = core
    N = core.ndim
    for n in range(N):
        G = grams[n]  # (R_n, R_n)
        tmp = cp.tensordot(G, tmp, axes=(1, n))  # new axis 0 is R_n
        tmp = cp.moveaxis(tmp, 0, n)
    tmp = cp.clip(tmp, a_min=epsilon, a_max=None)
    return tmp


def fr_compute_errors_largedim(
    vec_tensor,
    shape,
    core,
    factors,
    thread_budget=None,     # API compatibility; unused
    epsilon=1e-12,
    batch_rhat=1000,        # same role as in KL error
):
    """
    Relative Frobenius error ||X - X̂||_F / ||X||_F for sparse X,
    WITHOUT forming dense X̂.

    !! Still has some rounding differences compared to the original !!

    Uses:
      - ||X||_F^2 = sum_{nz} x^2
      - <X, X̂>   = sum_{nz} x * x̂  where x̂ computed at nz by Tucker contraction
      - ||X̂||_F^2 = <core, core ×_n (A_n^T A_n)> (exact, no dense X̂)
    """
    shape = tuple(int(s) for s in shape)
    N = len(shape)

    # --- decode NNZ (same helpers as your KL largedim) ---
    flat, x_nz = _blocked_coo_to_flat_indices(vec_tensor, shape)
    nnz = int(flat.size)

    x_nz = cp.asarray(x_nz)
    # Frobenius is fine with zeros, but keep nonneg pipeline consistent
    x_nz = cp.clip(x_nz, a_min=0.0, a_max=None)

    # --- ||X||_F ---
    norm_X_sq = cp.sum(x_nz * x_nz)
    norm_X = cp.sqrt(cp.maximum(norm_X_sq, epsilon))

    # Edge case: X is all zeros (relative error ill-defined); mirror your KL style
    if nnz == 0 or float(norm_X_sq.get()) == 0.0:
        # Return ||X̂||/max(||X||,eps) == ||X̂||/eps
        grams = [factors[n].T @ factors[n] for n in range(N)]
        Den = _core_multilinear_grams(core, grams, epsilon=epsilon)
        norm_Xhat_sq = cp.sum(core * Den)
        norm_Xhat = cp.sqrt(cp.maximum(norm_Xhat_sq, epsilon))
        return norm_Xhat / cp.maximum(norm_X, epsilon)

    idxs = _unravel_flat_indices_C(flat, shape)  # list of N arrays, each (nnz,)

    # --- compute xhat_nz in batches (same technique as KL error) ---
    inner_prod = cp.asarray(0.0, dtype=core.dtype)

    for start in range(0, nnz, int(batch_rhat)):
        end = min(start + int(batch_rhat), nnz)
        mats = [factors[n][idxs[n][start:end]] for n in range(N)]  # (b, Rn)
        xhat_b = trhat(core, mats, epsilon=epsilon)  # (b,)
        # <X, X̂> batch contribution
        inner_prod += cp.sum(x_nz[start:end] * xhat_b)

    # --- ||X̂||_F^2 exactly (no dense X̂, no mode_dot) ---
    grams = [factors[n].T @ factors[n] for n in range(N)]  # (R_n, R_n)
    Den = _core_multilinear_grams(core, grams, epsilon=epsilon)  # core-shaped
    norm_Xhat_sq = cp.sum(core * Den)

    # --- ||X - X̂||_F^2 = ||X||^2 + ||X̂||^2 - 2<X, X̂> ---
    residual_sq = norm_X_sq + norm_Xhat_sq - 2.0 * inner_prod
    residual_sq = cp.maximum(residual_sq, 0.0)
    residual_norm = cp.sqrt(residual_sq)

    return residual_norm / cp.maximum(norm_X, epsilon)


In [225]:
def fr_compute_errors(
        vec_tensor: cpx_sparse.spmatrix,
        shape,
        core,
        factors,
        thread_budget: ThreadBudget,
        epsilon=1e-12,
):
    """Relative Frobenius error ||X - X̂||_F / ||X||_F for sparse X.

    vec_tensor : vectorised sparse X (block-encoded, same as 'tensor')
    shape      : original N-D shape
    core       : current core G
    factors    : list of factor matrices A^{(n)}
    """

    shape = tuple(shape)

    # --- ||X||_F ---
    X_coo = vec_tensor.tocoo()
    x_nz = X_coo.data
    x_nz = tl.clip(x_nz, a_min=0.0, a_max=None)  # Frobenius is fine with zeros; keep nonneg pipeline consistent
    norm_X_sq = cp.sum(x_nz * x_nz)
    norm_X = cp.sqrt(cp.maximum(norm_X_sq, epsilon))

    # --- <X, X̂> = sum_{nz} X_i * X̂_i ---
    core_np = tl.to_numpy(core)
    factors_np = [tl.to_numpy(f) for f in factors]
    tucker = ptl.TuckerTensor(core=core_np, factors=factors_np)

    with thread_budget.limit():
        R_cpu = ptl_tucker_to_tensor(tucker)  # dense on CPU

    xhat_nz = gather_dense_at_block_nz(R_cpu, vec_tensor, shape)
    xhat_nz = cp.asarray(tl.clip(xhat_nz, a_min=epsilon, a_max=None))

    inner_prod = cp.sum(x_nz * xhat_nz)

    # --- ||X̂||_F^2 without forming X̂ ---
    # ||X̂||_F^2 = <G, G ×_n (A_n^T A_n)>
    denom = core
    for mode, A in enumerate(factors):
        AtA = tl.dot(tl.transpose(A), A)
        denom = mode_dot(denom, AtA, mode)

    denom = tl.clip(denom, a_min=epsilon, a_max=None)
    norm_Xhat_sq = cp.sum(core * denom)

    # --- ||X - X̂||_F^2 = ||X||_F^2 + ||X̂||_F^2 - 2<X, X̂> ---
    residual_sq = norm_X_sq + norm_Xhat_sq - 2.0 * inner_prod
    residual_sq = cp.maximum(residual_sq, 0.0)
    residual_norm = cp.sqrt(residual_sq)

    relative_error = residual_norm / norm_X
    return relative_error

In [226]:
batch_rhat = estimate_batch_rhat_for_tensordot(core, factors, safety=0.99, temp_mult=1.1)

fr_compute_errors_largedim(vec_tensor=sparse_tensor.tensor,
                    core=core_old,
                    factors=factors,
                    shape=shape,
                    thread_budget=thread_budget,
                    batch_rhat=batch_rhat,
                          )

array(0.999315, dtype=float32)

In [227]:
fr_compute_errors(vec_tensor=sparse_tensor.tensor,
                    core=core_old,
                    factors=factors,
                    shape=shape,
                    thread_budget=thread_budget,)

array(0.9993067, dtype=float32)

In [23]:
from tensormet.tucker_tensor import TuckerDecomposition
tk = TuckerDecomposition.load_from_disk(
    dataset=dataset,
    divergence=divergence,
    dims=2000,
    rank=150,
    iterations=500,
    map_location="cpu"

)

In [24]:
tk_core = cp.asarray(tk.core)

In [26]:
tk_core.max()

array(1785.3192, dtype=float32)

In [237]:
tk_factors = [cp.asarray(factor) for factor in tk.factors]

In [238]:
print(fr_compute_errors(vec_tensor=sparse_tensor.tensor,
                  core=tk_core,
                  factors=tk_factors,
                  shape=shape,
                  thread_budget=thread_budget, ))

406.75452


In [239]:
print(fr_compute_errors_largedim(vec_tensor=sparse_tensor.tensor,
                           core=tk_core,
                           factors=tk_factors,
                           shape=shape,
                           thread_budget=thread_budget,
                           batch_rhat=batch_rhat,
                           ))

406.7534
